# 展平层

在上一章，我们加载了 MNIST 数据集，并将每张图像处理成了 `(1, 28, 28)` 的三维数组：1 个通道，28 行，28 列。

但有一个问题：我们框架里的**线性层**，只能接受**向量**（一维数组）作为输入。面对多维的图像数据，它无从下手。

最直接的解决办法，是在线性层之前加一个**展平层**（Flatten Layer）：把无论多少维的数据，按顺序重新排列成一行，变成一个向量。

```{figure} images/flatten.png
:align: center
:width: 480px
**图例：展平层将多维图像数据拉平成一维向量**
```

以 MNIST 为例：一张 `(1, 28, 28)` 的图像，经过展平层后变成长度为 `784`（= 1×28×28）的向量，再送入线性层处理。

这章我们来实现展平层，并用它搭建第一个能识别手写数字的网络模型。

In [1]:
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### MNIST 数据集

在 MNIST 数据集里，我们实现一个评估函数 `estimate()` ，用来计算模型在测试集上的分类准确率。

模型的输出是一个长度为 10 的向量，向量中数值最大的位置，就代表模型认为最可能是的数字。我们用 `argmax()` 找到这个位置，再与真实标签的 `argmax()`（即单热向量中 1 所在的位置）比较，统计有多少预测是正确的。

例如，真实标签 $y$ = [0, 0, 1, 0, …, 0]，`argmax()` 结果为 2；模型输出 $p$ = [0.01, 0.02, 0.81, 0.05, …]，`argmax()` 结果也为 2，则预测正确。

In [8]:
class MNISTDataset(Dataset):

    def __init__(self, filename, batch_size=1):
        self.filename = filename
        super().__init__(batch_size)

    def load(self):
        with np.load(self.filename, allow_pickle=True) as f:
            self.train_features, self.train_labels = self.preprocess(f['x_train'], f['y_train'])
            self.test_features, self.test_labels = self.preprocess(f['x_test'], f['y_test'])

    @staticmethod
    def preprocess(x, y):
        inputs = x / 255.0
        inputs = np.expand_dims(inputs, axis=1)
        targets = np.zeros((len(y), 10))
        targets[np.arange(len(y)), y] = 1
        return inputs, targets

    def estimate(self, predictions):
        # 找到预测向量和标签向量中最大值的位置（即预测的数字），逐一比较
        predicted_digits = predictions.data.argmax(axis=1)
        digits = self.labels.argmax(axis=1)
        correct = (predicted_digits == digits).sum()
        return correct / len(self.labels)

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 展平层

展平层的实现非常简洁，核心就是 NumPy 的 `reshape()`。

**前向传播**：把输入张量从任意多维，重塑为二维 `(n, m)`，其中 `n` 是批大小，`m` 是其他所有维度的乘积。`reshape(n, -1)` 中的 `-1` 是 NumPy 的自动推断：只要指定第一维是 `n`，NumPy 会自动算出第二维应该是多少。

例如，输入形状为 `(2, 1, 28, 28)`，`reshape(2, -1)` 会得到 `(2, 784)`。

**梯度计算**：反向传播时，后一层传回来的梯度形状是 `(n, m)`，我们用同样的 `reshape()` 把它恢复成输入的原始形状，再往前传。展平层不改变数据本身，只改变排列方式，所以梯度也只需要**重塑形状**就够了。

**参数**：展平层没有任何可学习的参数，`parameters` 为空（继承自基础层）。

In [11]:
class Flatten(Layer):

    def forward(self, x):
        p = Tensor(x.data.reshape(x.data.shape[0], -1))

        def gradient_fn():
            x.grad += p.grad.reshape(x.data.shape)

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

### ReLU 激活函数层

In [12]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 均方误差损失函数

In [13]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / y.data.size

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 随机梯度下降优化器

In [14]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

In [15]:
class NNModel(Model):

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [16]:
LEARNING_RATE = 0.1

### 超参数：批大小

In [17]:
BATCH_SIZE = 2

### 超参数：轮次

In [18]:
EPOCHS = 10

### 建模

我们的第一个图像识别网络结构如下：

1. **展平层**：把 `(1, 28, 28)` 的图像拉平为长度 784 的向量；
2. **线性层（隐藏层）**：784 → 64 个神经元，后接 ReLU 激活函数；
3. **线性层（输出层）**：64 → 10 个神经元，对应 10 个数字类别。

In [19]:
dataset = MNISTDataset('tinymnist.npz', BATCH_SIZE)

layer = Sequential([Flatten(),
                    Linear(1 * 28 * 28, 64),
                    ReLU(),
                    Linear(64, 10)])
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Flatten[]
Linear[weight(64, 784); bias(64,)]
ReLU[]
Linear[weight(10, 64); bias(10,)]


### 迭代

In [20]:
model.train(dataset, EPOCHS)

## 验证

### 测试

训练完成后，切换到测试集，评估模型的分类准确率。

注意：测试时我们一次性传入全部测试数据 `dataset.items()`，而不是按批次迭代。测试阶段不需要反向传播，不存在内存压力的问题，使用全批量推理更方便。

In [21]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 92.00%


仅仅用展平层 + 两个线性层，不借助任何图像专用的结构，就能在 TinyMNIST 上达到 92% 的准确率，已经相当不错了。

不过这个模型有一个根本局限：展平操作破坏了图像的**空间结构**。像素之间的位置关系在被拉成一维向量的那一刻就丢失了，因此模型并不是真正看懂了图像的笔画，而是孤立地分析了每个像素的表现。

## 课后练习

在 `Flatten.forward` 中加几行 `print()`，打印出 `x.data.shape`（输入形状）和 `p.data.shape`（输出形状），运行一次训练，观察展平前后的形状变化。